# CREST msreact 反应网络生成（.xyz 构象）

本 Notebook 接收一个 CREST 的 `.xyz` 构象文件，通过 Python 调用 CREST 的 `msreact` 模块生成反应网络，并在输出目录中整理与展示关键结果。

- 全程在 Python 中完成调用（非 CLI 风格交互）
- 每一步都有清晰的分块与说明


## 1. 配置输入路径与计算参数

在此配置 CREST 可执行文件路径、输入 `.xyz` 文件路径、工作目录及电荷/自旋等参数。

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil

# ========== 用户配置区 ==========
crest_exe = None  # 例如: '/usr/local/bin/crest' 或 None 使用 PATH
xyz_path = Path('input.xyz')  # 替换为你的输入构象文件
work_dir = Path('crest_msreact_run')
charge = 0
uhf = 0
solvent = None  # 例如 'water'，如不使用溶剂设为 None
# ===============================

work_dir.mkdir(parents=True, exist_ok=True)
if crest_exe is None:
    crest_exe = shutil.which('crest')

if crest_exe is None:
    raise FileNotFoundError('未找到 crest 可执行文件，请检查环境或设置 crest_exe 路径。')

if not xyz_path.exists():
    raise FileNotFoundError(f'输入文件不存在: {xyz_path.resolve()}')

print('CREST 可执行文件:', crest_exe)
print('输入 xyz 文件:', xyz_path.resolve())
print('工作目录:', work_dir.resolve())

## 2. 读取与检查输入构象

读取 `.xyz` 文件并做基本检查（原子数、行数完整性），方便在提交计算前发现格式问题。

In [ ]:
xyz_lines = xyz_path.read_text(encoding='utf-8').strip().splitlines()
if len(xyz_lines) < 2:
    raise ValueError('XYZ 文件内容过短，请确认格式。')

try:
    n_atoms = int(xyz_lines[0].strip())
except ValueError as exc:
    raise ValueError('XYZ 文件首行应为原子数整数。') from exc

if len(xyz_lines) < n_atoms + 2:
    raise ValueError('XYZ 文件行数不足，疑似不完整。')

print(f'XYZ 原子数: {n_atoms}')
print('XYZ 文件基本检查通过。')

## 3. 调用 CREST 的 msreact 模块

将输入构象送入 CREST，并启用 `msreact` 进行反应网络搜索。运行日志保存在工作目录中，便于回溯。

In [ ]:
import subprocess

def run_msreact() -> None:
    cmd = [
        str(crest_exe),
        str(xyz_path.resolve()),
        '--msreact',
        '--chrg',
        str(charge),
        '--uhf',
        str(uhf),
    ]

    if solvent:
        cmd += ['--gbsa', solvent]

    log_path = work_dir / 'crest_msreact.log'
    print('运行命令:', ' '.join(cmd))

    with log_path.open('w', encoding='utf-8') as log_file:
        result = subprocess.run(
            cmd,
            cwd=work_dir,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )

    if result.returncode != 0:
        raise RuntimeError(f'CREST 运行失败，返回码: {result.returncode}。请查看日志: {log_path}')

    print('CREST msreact 运行完成，日志写入:', log_path.resolve())

run_msreact()

## 4. 收集并查看反应网络输出

扫描工作目录中的输出文件，识别可能的反应网络结果（例如 JSON、图文件或文本）。如果存在 JSON 结果，尝试读取并展示关键信息。

In [ ]:
import json

output_files = sorted(work_dir.glob('*'))
print('工作目录输出文件:')
for path in output_files:
    print('-', path.name)

# 尝试寻找 JSON 格式反应网络
json_candidates = [p for p in output_files if p.suffix.lower() == '.json']
if json_candidates:
    network_path = json_candidates[0]
    print(f'检测到 JSON 输出: {network_path.name}')
    data = json.loads(network_path.read_text(encoding='utf-8'))
    if isinstance(data, dict):
        print('JSON keys:', list(data.keys()))
    else:
        print('JSON 结构类型:', type(data))
else:
    print('未发现 JSON 输出，请查看日志或 CREST 文档确认输出格式。')